# Peta Perubahan Tutupan Lahan Kabupaten Banjar 2025 vs 2026 — Versi Planetary Computer (Tanpa Login)

**Mata kuliah:** Sistem Informasi Geografi (SIG) — **Dosen:** Dr. Ir. Mufidah Asy'ari, M.P.
**Wilayah:** Kabupaten Banjar, Kalimantan Selatan (luas BPS 4.668,50 km²)
**Citra:** Landsat 8 + Landsat 9, Collection 2 Level-2 Surface Reflectance, komposit median Juni–September 2025 vs 2026 (Microsoft Planetary Computer STAC)
**Metode:** Random Forest supervised (scikit-learn, 300 pohon), dilatih **pooled** pada komposit 2025 DAN 2026
**Fitur (14):** band B2–B7 + NDVI, NDWI, NDBI, EVI, NDMI, BSI + elevasi + kemiringan (Copernicus DEM 30 m)
**Kelas (5):** 1 Hutan · 2 Pertanian & Perkebunan · 3 Permukiman / Lahan Terbangun · 4 Badan Air & Rawa · 5 Lahan Terbuka & Semak Belukar

> **Versi ini TIDAK butuh login apa pun** — semua data diunduh dari Planetary Computer yang terbuka. Ada juga versi Google Earth Engine dengan metode yang sama: `gee_landcover_change_banjar.py` dan `landcover_change_banjar_GEE.ipynb` (butuh daftar di code.earthengine.google.com + Google Cloud project).

> **Jujur soal data:** tidak ada survei lapangan dalam tenggat tugas ini, jadi label latih/validasi diambil dari ESA WorldCover 2021 yang dipetakan ke 5 kelas. Akurasi yang dihitung adalah akurasi **terhadap referensi WorldCover 2021**, bukan akurasi lapangan. Sawit/karet di WorldCover umumnya tercatat sebagai "Tree cover", sehingga kelas Hutan cenderung terlalu luas dan Pertanian & Perkebunan terlalu sempit. Hasil yang dicapai: **OA 2025 = 0,679, Kappa = 0,599; OA 2026 = 0,689, Kappa = 0,611 — di bawah target OA ≥ 0,85**. Sudah dicoba: tambah DEM/indeks (v1 9 fitur OA 0,62 → v2 14 fitur OA 0,68), latih pooled, tambah sampel, ExtraTrees (lebih buruk). Lihat README untuk penjelasan lengkap.

## 1. Install package

Jalankan cell ini dulu. Bila Colab meminta restart runtime, lakukan **Runtime → Restart runtime**, lalu jalankan lagi dari cell ini ke bawah.

In [ ]:
!pip install -q pystac-client planetary-computer rasterio geopandas scikit-learn folium pillow
print("install selesai")

## 2. Konfigurasi

In [ ]:
import os, json, time, gc
import numpy as np
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.transform import Affine
from rasterio.features import rasterize

DATA, OUT = "data", "output"
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
BBOX = [114.5066, -3.7307, 115.5958, -2.8282]  # lon/lat Kab. Banjar
TAHUN_A, TAHUN_B = 2025, 2026
BANDS = ["blue", "green", "red", "nir08", "swir16", "swir22"]  # B2..B7
MAX_PER_TILE, N_PER_KELAS, N_POHON, SEED = 3, 2500, 300, 42
PURITY_MIN = 226   # kemurnian >= 8/9 (disimpan 0-255)
LUAS_BPS_KM2 = 4668.50
KELAS = {1: "Hutan", 2: "Pertanian & Perkebunan",
         3: "Permukiman / Lahan Terbangun", 4: "Badan Air & Rawa",
         5: "Lahan Terbuka & Semak Belukar"}
WC_MAP = {10: 1, 95: 1, 40: 2, 50: 3, 80: 4, 90: 4, 20: 5, 30: 5, 60: 5}
CRS = "EPSG:32650"
print("konfigurasi siap")

## 3. Batas Kabupaten Banjar (geoBoundaries ADM2, validasi vs BPS)

In [ ]:
import geopandas as gpd, requests, io
url = ("https://github.com/wmgeolab/geoBoundaries/raw/9469f09/"
       "releaseData/gbOpen/IDN/ADM2/geoBoundaries-IDN-ADM2_simplified.geojson")
r = requests.get(url, timeout=300); r.raise_for_status()
g = gpd.read_file(io.BytesIO(r.content))
b = g[g["shapeName"] == "Banjar"].copy()  # hanya ada satu 'Banjar' di ADM2
b.to_file(f"{DATA}/banjar_boundary.geojson", driver="GeoJSON")
luas = b.to_crs(CRS).geometry.area.iloc[0] / 1e6
print(f"Luas batas Banjar: {luas:,.2f} km2 (BPS {LUAS_BPS_KM2:,.2f} km2, "
      f"selisih {abs(luas-LUAS_BPS_KM2)/LUAS_BPS_KM2*100:.2f}%) — toleransi 5%")
batas_path = f"{DATA}/banjar_boundary.geojson"

def grid_target():
    bb = b.to_crs(CRS)
    minx, miny, maxx, maxy = bb.total_bounds
    minx = np.floor(minx/30)*30; miny = np.floor(miny/30)*30
    maxx = np.ceil(maxx/30)*30;  maxy = np.ceil(maxy/30)*30
    w, h = int((maxx-minx)/30), int((maxy-miny)/30)
    return w, h, Affine(30, 0, minx, 0, -30, maxy)
W, H, TRANSFORM = grid_target()
print(f"Grid target: {W} x {H} piksel @ 30 m")

## 4. Cari scene Landsat di STAC Planetary Computer

In [ ]:
from collections import defaultdict
from pystac_client import Client

def cari_scene(tahun):
    cat = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    items = list(cat.search(collections=["landsat-c2-l2"], bbox=BBOX,
        datetime=f"{tahun}-06-01/{tahun}-09-30",
        query={"eo:cloud_cover": {"lt": 20}}).items())
    print(f"{tahun}: {len(items)} scene cloud<20")
    return items

def pilih_scene(items):
    by = defaultdict(list)
    for it in items:
        if "_T2" in it.id:
            continue
        by[it.id.split("_")[2]].append(it)
    out = []
    for tile in sorted(by):
        v = sorted(by[tile], key=lambda x: x.properties["eo:cloud_cover"])
        out.extend(v[:MAX_PER_TILE])
    return out

items_2025, items_2026 = cari_scene(2025), cari_scene(2026)
print("scene dipilih 2025:", len(pilih_scene(items_2025)))
print("scene dipilih 2026:", len(pilih_scene(items_2026)))

## 5. Komposit median Jun–Sep (mask awan QA_PIXEL)

Cell ini paling lama (sekitar 15 menit per tahun di Colab) karena membaca citra langsung dari Planetary Computer per blok baris. Biarkan berjalan sampai selesai.

In [ ]:
import planetary_computer
from pystac import Item

def bangun_komposit(tahun, items):
    out = f"{DATA}/komposit_{tahun}.tif"
    if os.path.exists(out):
        print(f"Komposit {tahun} sudah ada, lewati ({out})")
        return
    scenes = [planetary_computer.sign(Item.from_dict(it.to_dict()))
              for it in pilih_scene(items)]
    print(f"{tahun}: {len(scenes)} scene dipakai")
    prof = dict(driver="GTiff", width=W, height=H, count=len(BANDS),
                dtype="uint16", crs=CRS, transform=TRANSFORM, nodata=0,
                compress="deflate", tiled=True, blockxsize=256, blockysize=256)
    blok = 256
    with rasterio.open(out, "w", **prof) as dst:
        for bi, band in enumerate(BANDS, start=1):
            print(f"[{tahun}] band {band} ({bi}/6) ...", flush=True)
            peg = []
            for item in scenes:
                peg.append((rasterio.open(item.assets[band].href),
                            rasterio.open(item.assets["qa_pixel"].href)))
            for r0 in range(0, H, blok):
                rh = min(blok, H - r0)
                x0, y_top = TRANSFORM * (0, r0)
                x1, y_bot = TRANSFORM * (W, r0 + rh)
                bb = (x0, y_bot, x1, y_top)
                tumpuk = []
                for ds_b, ds_q in peg:
                    try:
                        wb = from_bounds(*bb, ds_b.transform)
                        wq = from_bounds(*bb, ds_q.transform)
                        if wb.width <= 0 or wb.height <= 0:
                            continue
                        arr = ds_b.read(1, window=wb, out_shape=(rh, W),
                                        resampling=rasterio.enums.Resampling.nearest)
                        qa = ds_q.read(1, window=wq, out_shape=(rh, W),
                                       resampling=rasterio.enums.Resampling.nearest)
                    except Exception:
                        continue
                    buruk = (arr == 0) | ((qa & 0b11110) != 0)  # bit 1-4 QA
                    a = arr.astype(np.float32); a[buruk] = np.nan
                    tumpuk.append(a)
                if tumpuk:
                    with np.errstate(all="ignore"):
                        med = np.nanmedian(np.stack(tumpuk), axis=0)
                    hasil = np.where(np.isnan(med), 0, med).astype(np.uint16)
                else:
                    hasil = np.zeros((rh, W), dtype=np.uint16)
                dst.write(hasil, bi, window=((r0, r0 + rh), (0, W)))
                del tumpuk, hasil
            for ds_b, ds_q in peg:
                ds_b.close(); ds_q.close()
    print(f"Komposit {tahun} tersimpan: {out}")

bangun_komposit(2025, items_2025)
bangun_komposit(2026, items_2026)

## 6. ESA WorldCover 2021 → 5 kelas (resample ke 30 m)

In [ ]:
from rasterio.warp import Resampling, reproject

def resample_worldcover():
    out = f"{DATA}/worldcover_30m.tif"
    if os.path.exists(out):
        print(f"WorldCover 30 m sudah ada, lewati ({out})")
        return
    cat = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    items = list(cat.search(collections=["esa-worldcover"], bbox=BBOX,
                            datetime="2021-01-01/2021-12-31").items())
    print(f"{len(items)} tile WorldCover 2021")
    srcs = [rasterio.open(planetary_computer.sign(it).assets["map"].href)
            for it in items]
    with rasterio.open(f"{DATA}/komposit_2025.tif") as comp:
        prof = comp.profile.copy(); prof.update(count=1, dtype="uint8", nodata=0)
        with rasterio.open(out, "w", **prof) as dst:
            for r0 in range(0, H, 256):
                rh = min(256, H - r0)
                win = Window(0, r0, W, rh)
                gabung = np.zeros((rh, W), dtype=np.uint8)
                tr = rasterio.windows.transform(win, TRANSFORM)
                for ds in srcs:
                    tmp = np.zeros((rh, W), dtype=np.uint8)
                    reproject(source=rasterio.band(ds, 1), destination=tmp,
                              src_transform=ds.transform, src_crs=ds.crs,
                              dst_transform=tr, dst_crs=CRS,
                              resampling=Resampling.mode,
                              src_nodata=0, dst_nodata=0)
                    isi = (gabung == 0) & (tmp != 0)
                    gabung[isi] = tmp[isi]
                peta = np.zeros((rh, W), dtype=np.uint8)
                for wc, kl in WC_MAP.items():
                    peta[gabung == wc] = kl
                dst.write(peta, 1, window=win)
    for ds in srcs:
        ds.close()
    print("WorldCover 30 m tersimpan:", out)

resample_worldcover()

## 7. DEM Copernicus 30 m + kemiringan (slope)

Elevasi dan kemiringan adalah dua fitur terpenting pada model v2 (kepentingan fitur 0,20 dan 0,16). Nilai DEM negatif (artefak di atas air) dipangkas ke 0 saat dipakai sebagai fitur.

In [ ]:
def dem_dan_kemiringan():
    if os.path.exists(f"{DATA}/dem_30m.tif") and os.path.exists(f"{DATA}/slope_30m.tif"):
        print("DEM + slope sudah ada, lewati")
        return
    cat = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    items = list(cat.search(collections=["cop-dem-glo-30"], bbox=BBOX).items())
    print(f"{len(items)} tile Copernicus DEM")
    srcs = [rasterio.open(planetary_computer.sign(it).assets["data"].href)
            for it in items]
    with rasterio.open(f"{DATA}/komposit_2025.tif") as comp:
        prof = comp.profile.copy()
        prof.update(count=1, dtype="float32", nodata=-9999)
        with rasterio.open(f"{DATA}/dem_30m.tif", "w", **prof) as dst:
            for r0 in range(0, H, 256):
                rh = min(256, H - r0)
                win = Window(0, r0, W, rh)
                gab = np.full((rh, W), -9999, dtype=np.float32)
                tr = rasterio.windows.transform(win, TRANSFORM)
                for ds in srcs:
                    tmp = np.full((rh, W), -9999, dtype=np.float32)
                    reproject(source=rasterio.band(ds, 1), destination=tmp,
                              src_transform=ds.transform, src_crs=ds.crs,
                              dst_transform=tr, dst_crs=CRS,
                              resampling=Resampling.bilinear,
                              src_nodata=-9999, dst_nodata=-9999)
                    isi = (gab == -9999) & (tmp != -9999)
                    gab[isi] = tmp[isi]
                dst.write(gab, 1, window=win)
    for ds in srcs:
        ds.close()
    with rasterio.open(f"{DATA}/dem_30m.tif") as ds:
        dem_arr = ds.read(1); prof2 = ds.profile.copy()
    dem_arr = np.where(dem_arr == -9999, np.nan, dem_arr)
    gy, gx = np.gradient(dem_arr, 30, 30)
    slope = np.degrees(np.arctan(np.sqrt(gx**2 + gy**2)))
    slope = np.where(np.isnan(slope), 0, slope).astype(np.float32)
    with rasterio.open(f"{DATA}/slope_30m.tif", "w", **prof2) as dst:
        dst.write(slope, 1)
    print("DEM + slope tersimpan")

dem_dan_kemiringan()

## 8. Peta kemurnian WorldCover

Untuk tiap sel 30 m, dihitung fraksi dari 9 piksel WorldCover 10 m yang sama dengan kelas modus di dalam sel itu (disimpan 0–255; 255 = 9/9 murni). Sampel latih hanya diambil dari piksel dengan kemurnian ≥ 8/9, supaya label latih tidak tercampur di tepi kelas.

In [ ]:
from rasterio.transform import Affine as _Af

def peta_kemurnian():
    if os.path.exists(f"{DATA}/wc_purity.tif"):
        print("Peta kemurnian sudah ada, lewati")
        return
    cat = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    items = list(cat.search(collections=["esa-worldcover"], bbox=BBOX,
                            datetime="2021-01-01/2021-12-31").items())
    srcs = [rasterio.open(planetary_computer.sign(it).assets["map"].href)
            for it in items]
    with rasterio.open(f"{DATA}/komposit_2025.tif") as comp:
        prof = comp.profile.copy()
        prof.update(count=1, dtype="uint8", nodata=0)
        with rasterio.open(f"{DATA}/wc_purity.tif", "w", **prof) as dst:
            for r0 in range(0, H, 256):
                rh = min(256, H - r0)
                win = Window(0, r0, W, rh)
                t = rasterio.windows.transform(win, TRANSFORM)
                gab = np.zeros((rh*3, W*3), dtype=np.uint8)
                t10 = _Af(t.a/3, t.b, t.c, t.d, t.e/3, t.f)
                for ds in srcs:
                    tmp = np.zeros((rh*3, W*3), dtype=np.uint8)
                    reproject(source=rasterio.band(ds, 1), destination=tmp,
                              src_transform=ds.transform, src_crs=ds.crs,
                              dst_transform=t10, dst_crs=CRS,
                              dst_nodata=0, src_nodata=0,
                              resampling=Resampling.nearest)
                    isi = (gab == 0) & (tmp != 0)
                    gab[isi] = tmp[isi]
                # 9 sub-piksel 10 m tiap sel 30 m (transpose agar urutan
                # sumbu benar sebelum reshape, lalu ratakan ke 9)
                a = gab.reshape(rh, 3, W, 3).transpose(0, 2, 1, 3).reshape(rh, W, 9)
                srt = np.sort(a, axis=2)
                same = np.ones((rh, W, 9), dtype=bool)
                same[:, :, 1:] = srt[:, :, 1:] == srt[:, :, :-1]
                maxrun = np.ones((rh, W), dtype=np.uint8)
                cur = np.ones((rh, W), dtype=np.uint8)
                for j in range(1, 9):
                    cur = np.where(same[:, :, j], cur + 1, 1).astype(np.uint8)
                    maxrun = np.maximum(maxrun, cur)
                frac = (maxrun.astype(np.uint16) * 255 // 9).astype(np.uint8)
                dst.write(frac, 1, window=win)
    for ds in srcs:
        ds.close()
    print("Peta kemurnian tersimpan: data/wc_purity.tif")

peta_kemurnian()

## 9. Training sample + Random Forest pooled + akurasi

Model **pooled**: lokasi sampel yang sama dipakai dua kali — sekali dengan fitur 2025, sekali dengan fitur 2026 — sehingga satu model stabil untuk kedua tahun (pada v1, model yang hanya dilatih pada 2025 runtuh saat dipakai ke 2026: OA 0,62 → 0,53).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split

def fitur(comp, dem_arr, slope_arr):
    r = comp.astype(np.float32) * 0.0000275 - 0.2
    blue, green, red, nir, swir1, swir2 = r
    eps = 1e-6
    ndvi = (nir-red)/(nir+red+eps)
    ndwi = (green-nir)/(green+nir+eps)
    ndbi = (swir1-nir)/(swir1+nir+eps)
    evi = 2.5*(nir-red)/(nir + 6*red - 7.5*blue + 1 + eps)
    ndmi = (nir-swir1)/(nir+swir1+eps)
    bsi = ((swir1+red)-(nir+blue))/((swir1+red)+(nir+blue)+eps)
    elev = np.clip(dem_arr, 0, None).astype(np.float32)
    return np.stack([blue, green, red, nir, swir1, swir2,
                     ndvi, ndwi, ndbi, evi, ndmi, bsi, elev, slope_arr])

def fitur_titik(comp, dem_arr, slope_arr, yy, xx):
    """14 fitur HANYA di titik sampel — hemat memori (aritmetika sama
    persis dengan fitur(), tapi dihitung pada (6, N) nilai sampel saja,
    bukan pada grid penuh 14 band float32 yang butuh ~750 MB)."""
    r = comp[:, yy, xx].astype(np.float32) * 0.0000275 - 0.2
    blue, green, red, nir, swir1, swir2 = r
    eps = 1e-6
    ndvi = (nir-red)/(nir+red+eps)
    ndwi = (green-nir)/(green+nir+eps)
    ndbi = (swir1-nir)/(swir1+nir+eps)
    evi = 2.5*(nir-red)/(nir + 6*red - 7.5*blue + 1 + eps)
    ndmi = (nir-swir1)/(nir+swir1+eps)
    bsi = ((swir1+red)-(nir+blue))/((swir1+red)+(nir+blue)+eps)
    elev = np.clip(dem_arr[yy, xx], 0, None).astype(np.float32)
    return np.stack([blue, green, red, nir, swir1, swir2,
                     ndvi, ndwi, ndbi, evi, ndmi, bsi,
                     elev, slope_arr[yy, xx]], axis=1).astype(np.float32)

with rasterio.open(f"{DATA}/komposit_2025.tif") as ds:
    c25 = ds.read()
with rasterio.open(f"{DATA}/komposit_2026.tif") as ds:
    c26 = ds.read()
with rasterio.open(f"{DATA}/worldcover_30m.tif") as ds:
    wc = ds.read(1)
with rasterio.open(f"{DATA}/wc_purity.tif") as ds:
    purity = ds.read(1)
with rasterio.open(f"{DATA}/dem_30m.tif") as ds:
    dem = ds.read(1)
with rasterio.open(f"{DATA}/slope_30m.tif") as ds:
    slope = ds.read(1)
shapes = [(g, 1) for g in gpd.read_file(batas_path).to_crs(CRS).geometry]
mask = rasterize(shapes, out_shape=wc.shape, transform=TRANSFORM,
                 fill=0, dtype=np.uint8)
valid = (mask == 1) & (wc > 0) & (purity >= PURITY_MIN) \
        & (c25[0] > 0) & (c26[0] > 0)
print(f"Piksel valid bersama (murni >=8/9): {valid.sum():,}")
rng = np.random.default_rng(SEED)
ys, xs = [], []
for k in sorted(KELAS):
    yy, xx = np.where(valid & (wc == k))
    print(f"kandidat kelas {k} {KELAS[k]}: {len(yy):,}")
    idx = rng.choice(len(yy), size=min(N_PER_KELAS, len(yy)), replace=False)
    ys.append(yy[idx]); xs.append(xx[idx])
ys = np.concatenate(ys); xs = np.concatenate(xs)
y = wc[ys, xs]
print("Total lokasi sampel:", len(y))
i_tr, i_va = train_test_split(np.arange(len(y)), test_size=0.30,
                              random_state=SEED, stratify=y)
y_tr, y_va = y[i_tr], y[i_va]

def fitur_sampel(comp, yy, xx):
    return fitur_titik(comp, dem, slope, yy, xx)
X_tr = np.vstack([fitur_sampel(c25, ys[i_tr], xs[i_tr]),
                  fitur_sampel(c26, ys[i_tr], xs[i_tr])])
y_tr2 = np.concatenate([y_tr, y_tr])
X_va25 = fitur_sampel(c25, ys[i_va], xs[i_va])
X_va26 = fitur_sampel(c26, ys[i_va], xs[i_va])
del c25, c26
gc.collect()
print(f"Baris latih pooled: {len(X_tr)} (14 fitur)")
clf = RandomForestClassifier(n_estimators=N_POHON, n_jobs=-1, random_state=SEED)
clf.fit(X_tr, y_tr2)
print("Random Forest selesai dilatih.")
print("Kepentingan fitur:", np.round(clf.feature_importances_, 3))

def laporan(y_true, y_pred, nama):
    oa = accuracy_score(y_true, y_pred); kap = cohen_kappa_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
    print(f"\nAkurasi {nama}: OA={oa:.4f}  Kappa={kap:.4f}")
    print("Confusion matrix (baris=referensi, kolom=prediksi):\n", cm)
    prod, user = [], []
    for i in range(5):
        p = cm[i, i]/cm[i, :].sum(); u = cm[i, i]/cm[:, i].sum()
        prod.append(float(p)); user.append(float(u))
        print(f"  kelas {i+1} {KELAS[i+1]:30s} Producer {p:.3f}  User {u:.3f}")
    return {"OA": float(oa), "Kappa": float(kap),
            "confusion_matrix": cm.tolist(),
            "producers": prod, "users": user}
stat = {"akurasi_2025": laporan(y_va, clf.predict(X_va25), "validasi 2025 (model pooled)"),
        "akurasi_2026": laporan(y_va, clf.predict(X_va26), "validasi 2026 (model pooled)")}

## 10. Prediksi peta 2025 & 2026, luas, dan matriks transisi

Prediksi memakai **mask bersama**: hanya piksel yang valid di KEDUA tahun yang diklasifikasi, supaya luas total kedua tahun sama dan matriks transisi konsisten. Prediksi per blok 256 baris agar hemat memori.

In [ ]:
common = (mask == 1)
with rasterio.open(f"{DATA}/komposit_2025.tif") as ds:
    common &= (ds.read(1) > 0)
with rasterio.open(f"{DATA}/komposit_2026.tif") as ds:
    common &= (ds.read(1) > 0)
print(f"Piksel terklasifikasi (mask bersama): {common.sum():,}")
prof = dict(driver="GTiff", width=W, height=H, count=1, dtype="uint8",
            crs=CRS, transform=TRANSFORM, nodata=0, compress="deflate",
            tiled=True, blockxsize=256, blockysize=256)
for tahun in (2025, 2026):
    out = f"{OUT}/klasifikasi_{tahun}.tif"
    print(f"Prediksi {tahun} ...", flush=True)
    with rasterio.open(f"{DATA}/komposit_{tahun}.tif") as ds, \
         rasterio.open(out, "w", **prof) as dst:
        for r0 in range(0, H, 256):
            rh = min(256, H - r0)
            comp = ds.read(window=Window(0, r0, W, rh))
            f = fitur(comp, dem[r0:r0+rh], slope[r0:r0+rh])
            del comp
            X = np.moveaxis(f, 0, -1).reshape(-1, f.shape[0])
            del f
            gc.collect()
            mflat = common[r0:r0+rh].reshape(-1)
            pred = np.zeros(rh * W, dtype=np.uint8)
            idx = np.where(mflat)[0]
            for s in range(0, len(idx), 100000):
                chunk = idx[s:s+100000]
                pred[chunk] = clf.predict(X[chunk])
                del chunk
            del X, idx
            gc.collect()
            dst.write(pred.reshape(rh, W), 1, window=Window(0, r0, W, rh))
    print(" tersimpan", out)

with rasterio.open(f"{OUT}/klasifikasi_2025.tif") as ds:
    k25 = ds.read(1)
with rasterio.open(f"{OUT}/klasifikasi_2026.tif") as ds:
    k26 = ds.read(1)
kode = np.where((k25 > 0) & (k26 > 0), k25*10 + k26, 0).astype(np.uint8)
with rasterio.open(f"{OUT}/perubahan_2025_2026.tif", "w", **prof) as dst:
    dst.write(kode, 1)
px_ha = 30 * 30 / 1e4
luas = {}
for tahun, k in ((2025, k25), (2026, k26)):
    luas[str(tahun)] = {KELAS[c]: float((k == c).sum() * px_ha) for c in KELAS}
    luas[str(tahun)]["TOTAL"] = float((k > 0).sum() * px_ha)
trans = {(a, b): float(((k25 == a) & (k26 == b)).sum() * px_ha)
         for a in KELAS for b in KELAS}
stat["luas_ha"] = luas
stat["matriks_transisi_ha"] = {f"{a}->{b}": v for (a, b), v in trans.items()}
top = sorted(((v, a, b) for (a, b), v in trans.items() if a != b),
             reverse=True)[:3]
stat["transisi_terbesar_ha"] = [
    {"dari": KELAS[a], "ke": KELAS[b], "ha": float(v)} for v, a, b in top]
print("\nLuas (ha):")
for t in ("2025", "2026"):
    print(t, {k: round(v, 1) for k, v in luas[t].items()})
print("3 transisi perubahan terbesar:")
for v, a, b in top:
    print(f"  {KELAS[a]} -> {KELAS[b]}: {v:,.1f} ha")
with open(f"{OUT}/statistik.json", "w") as f:
    json.dump(stat, f, indent=2, ensure_ascii=False)
print("Statistik tersimpan: output/statistik.json")

## 11. Tampilkan peta (folium, overlay resolusi 120 m)

In [ ]:
import folium
from PIL import Image
from rasterio.warp import transform_bounds

PALET = {1: "#1a9850", 2: "#fee08b", 3: "#d73027", 4: "#2166ac", 5: "#bdbdbd"}
lon_min, lat_min, lon_max, lat_max = transform_bounds(CRS, "EPSG:4326",
    TRANSFORM[2], TRANSFORM[5] + TRANSFORM[4]*H,
    TRANSFORM[2] + TRANSFORM[0]*W, TRANSFORM[5])
def kecilkan(k):
    return k[::4, ::4]
def png_overlay(k, path, cmap_dict):
    h2, w2 = k.shape
    rgb = np.zeros((h2, w2, 4), dtype=np.uint8)
    for c, hexcol in cmap_dict.items():
        r, g, b = int(hexcol[1:3], 16), int(hexcol[3:5], 16), int(hexcol[5:7], 16)
        m = k == c
        rgb[m] = [r, g, b, 255]
    Image.fromarray(rgb, "RGBA").save(path)

png_overlay(kecilkan(k25), f"{OUT}/peta_2025.png", PALET)
png_overlay(kecilkan(k26), f"{OUT}/peta_2026.png", PALET)
# peta perubahan: hanya piksel berubah (kode a!=b), warna = kelas tujuan
kode_kecil = kecilkan(kode)
rgb = np.zeros((*kode_kecil.shape, 4), dtype=np.uint8)
for a in KELAS:
    for b in KELAS:
        if a == b:
            continue
        m = kode_kecil == a*10 + b
        hexcol = PALET[b]
        r, g, bl = int(hexcol[1:3], 16), int(hexcol[3:5], 16), int(hexcol[5:7], 16)
        rgb[m] = [r, g, bl, 255]
Image.fromarray(rgb, "RGBA").save(f"{OUT}/peta_perubahan.png")

m = folium.Map(location=[-3.32, 115.05], zoom_start=10, tiles="OpenStreetMap")
bounds = [[lat_min, lon_min], [lat_max, lon_max]]
folium.raster_layers.ImageOverlay(image=f"{OUT}/peta_2025.png", bounds=bounds,
    opacity=0.85, name="Klasifikasi 2025").add_to(m)
folium.raster_layers.ImageOverlay(image=f"{OUT}/peta_2026.png", bounds=bounds,
    opacity=0.85, name="Klasifikasi 2026").add_to(m)
folium.raster_layers.ImageOverlay(image=f"{OUT}/peta_perubahan.png", bounds=bounds,
    opacity=0.85, name="Perubahan 2025-2026").add_to(m)
folium.LayerControl().add_to(m)
m

## 12. Unduh semua hasil (zip)

In [ ]:
import shutil
shutil.make_archive("hasil_gis_banjar", "zip", OUT)
try:
    from google.colab import files
    print("Mengunduh hasil_gis_banjar.zip ...")
    files.download("hasil_gis_banjar.zip")
except ImportError:
    # Bukan di Colab (mis. Jupyter lokal): zip sudah tersimpan di folder kerja.
    print("Bukan di Colab — file hasil_gis_banjar.zip sudah tersimpan di folder kerja.")